# EDA v2

Uses the unified 11-band v4 TIFs exported from GEE at 500m / EPSG:4326.
Each country is a single file with bands:

| # | Band | Role |
|---|------|------|
| 0 | viirs_2013 | input |
| 1 | viirs_2014 | input |
| 2 | viirs_2015 | input |
| 3 | viirs_2016 | input |
| 4 | builtup_fraction | input |
| 5 | population | input |
| 6 | elevation | input |
| 7 | slope | input |
| 8 | infra_proximity | input |
| 9 | smod_code | input |
| 10 | viirs_2017 | **target only** |

~42% of pixels per country are NaN — these are pixels outside the country polygon within the bounding-box raster. All functions use `np.nanmean` and skip patches with >50% NaN.

## Imports and constants

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR  = Path("data/gee/v4")
PATCH_DIR = Path("data/patches_v4")

COUNTRIES = [
    "Albania",
    "Bosnia_and_Herzegovina",
    "Kosovo",
    "Montenegro",
    "North_Macedonia",
    "Serbia",
]

BAND_NAMES = [
    "viirs_2013", "viirs_2014", "viirs_2015", "viirs_2016",
    "builtup_fraction", "population",
    "elevation", "slope", "infra_proximity",
    "smod_code",
    "viirs_2017",
]

PATCH_SIZE   = 32
STRIDE       = 16
BLANK_THRESH = 0.3    # min nanmean of max(viirs_2013, viirs_2017) to keep a patch
NAN_THRESH   = 0.5   # max NaN fraction per patch before skipping


## Inspect TIFs

In [ ]:
rows = []
for country in COUNTRIES:
    path = DATA_DIR / f"balkans_{country}_v4.tif"
    with rasterio.open(path) as src:
        arr  = src.read().astype(np.float32)
        prof = src.profile
    for i, name in enumerate(BAND_NAMES):
        rows.append({
            "country": country,
            "band":    name,
            "height":  arr.shape[1],
            "width":   arr.shape[2],
            "crs":     prof["crs"].to_epsg(),
            "nan%":    round(float(np.isnan(arr[i]).mean() * 100), 1),
            "min":     round(float(np.nanmin(arr[i])), 4),
            "max":     round(float(np.nanmax(arr[i])), 4),
        })

df = pd.DataFrame(rows)
pd.set_option("display.max_rows", 80)
print(df.to_string(index=False))


## Load country

In [ ]:
def load_country(country: str) -> dict:
    """
    Read the unified 11-band v4 TIF. Pixels outside the country polygon are NaN.
    Band order: ch0-3=VIIRS 2013-2016, ch4=builtup, ch5=pop, ch6=elev,
                ch7=slope, ch8=infra_prox, ch9=smod_code, ch10=viirs_2017 (target).
    """
    path = DATA_DIR / f"balkans_{country}_v4.tif"
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)

    viirs_input = np.clip(data[0:4], 0, None)         # (4, H, W) 2013-2016
    builtup     = np.clip(data[4:5], 0, 1)            # (1, H, W) fraction
    pop         = np.where(data[5:6] < 0, np.nan, data[5:6])
    elev        = data[6:7]                           # (1, H, W) metres
    slope       = data[7:8]                           # (1, H, W) degrees
    prox        = np.clip(data[8:9], 0, None)         # (1, H, W) pixel dist
    smod        = data[9:10]                          # (1, H, W) settlement class
    viirs_2017  = np.clip(data[10:11], 0, None)       # (1, H, W) target

    return {
        "viirs_input": viirs_input,
        "viirs_2017":  viirs_2017,
        "builtup":     builtup,
        "pop":         pop,
        "elev":        elev,
        "slope":       slope,
        "prox":        prox,
        "smod":        smod,
    }


# sanity check
test = load_country("Kosovo")
for k, v in test.items():
    nan_pct = np.isnan(v).mean() * 100
    print(f"{k:<15} {v.shape}  nan={nan_pct:.1f}%  min={np.nanmin(v):.3f}  max={np.nanmax(v):.3f}")


## Visualise raw layers per country

In [ ]:
def plot_country(country: str) -> None:
    d = load_country(country)

    fig, axes = plt.subplots(2, 4, figsize=(22, 9))
    fig.suptitle(country.replace("_", " "), fontsize=14, fontweight="bold")

    viirs_2013_disp = np.log1p(np.clip(d["viirs_input"][0], 0, None))
    viirs_2017_disp = np.log1p(np.clip(d["viirs_2017"][0], 0, None))

    panels = [
        (viirs_2013_disp,                               "magma",   "VIIRS 2013 (log1p display)"),
        (viirs_2017_disp,                               "magma",   "VIIRS 2017 (log1p display)"),
        (d["builtup"][0],                               "YlOrBr",  "Built-up fraction"),
        (np.log1p(np.nan_to_num(d["pop"][0], nan=0)),  "YlOrRd",  "Population (log1p)"),
        (d["elev"][0],                                  "terrain", "Elevation (m)"),
        (d["slope"][0],                                 "copper",  "Slope (deg)"),
        (np.log1p(np.nan_to_num(d["prox"][0], nan=0)), "Blues",   "Infra proximity (log1p)"),
        (d["smod"][0],                                  "tab20",   "SMOD code"),
    ]

    for ax, (arr, cmap, title) in zip(axes.flat, panels):
        valid = arr[np.isfinite(arr)]
        vmin = np.percentile(valid, 1)
        vmax = np.percentile(valid, 99)
        im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.tight_layout()
    plt.show()


for country in COUNTRIES:
    plot_country(country)


## Nightlight change maps (2013 → 2017)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Nightlight % change 2013–2017", fontsize=14, fontweight="bold")

for ax, country in zip(axes.flat, COUNTRIES):
    d          = load_country(country)
    y2013      = d["viirs_input"][0]
    y2017      = d["viirs_2017"][0]
    pct_change = (y2017 - y2013) / (y2013 + 1e-6) * 100

    valid = pct_change[np.isfinite(pct_change)]
    vmax  = np.percentile(np.abs(valid), 95)
    im    = ax.imshow(pct_change, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, label="%")
    ax.set_title(country.replace("_", " "))
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle("Nightlight targets: log-ratio and departure from momentum", fontsize=14, fontweight="bold")

for col, country in enumerate(COUNTRIES):
    d   = load_country(country)
    vi  = d["viirs_input"]   # (4, H, W)
    v17 = d["viirs_2017"]    # (1, H, W)

    log_ratio = np.log1p(v17[0]) - np.log1p(vi[0])

    means_log = np.array([np.log1p(vi[t]) for t in range(4)])   # (4, H, W)
    slope = (means_log[3] - means_log[0]) / 3.0
    predicted_log = means_log[3] + slope
    actual_log = np.log1p(v17[0])
    departure_from_momentum = actual_log - predicted_log

    targets = [
        ("Log-ratio", log_ratio),
        ("Momentum", departure_from_momentum),
    ]

    for row, (name, arr) in enumerate(targets):
        ax = axes[row, col]
        valid = arr[np.isfinite(arr)]
        vmax = np.percentile(np.abs(valid), 95)
        im = ax.imshow(arr, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        plt.colorbar(im, ax=ax, fraction=0.046)
        ax.set_title(f"{country.replace('_', ' ')}\n{name}")
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle("Patch terciles over VIIRS background", fontsize=14, fontweight="bold")

label_cmap = ListedColormap(["steelblue", "goldenrod", "firebrick"])
label_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], label_cmap.N)

for col, country in enumerate(COUNTRIES):
    d   = load_country(country)
    vi  = d["viirs_input"]   # (4, H, W)
    v17 = d["viirs_2017"]    # (1, H, W)

    H, W = vi.shape[1], vi.shape[2]

    bg = np.log1p(np.clip(vi[0], 0, None))
    bg_valid = bg[np.isfinite(bg)]
    bg_vmin = np.percentile(bg_valid, 1)
    bg_vmax = np.percentile(bg_valid, 99)

    log_ratio = np.log1p(v17[0]) - np.log1p(vi[0])

    patch_coords = []
    patch_logratio = []
    patch_momentum = []

    for y in range(0, H - PATCH_SIZE, STRIDE):
        for x in range(0, W - PATCH_SIZE, STRIDE):
            v17_patch = v17[0, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            if np.isnan(v17_patch).mean() > NAN_THRESH:
                continue

            vi2013_patch = vi[0, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            if np.nanmean(vi2013_patch) < BLANK_THRESH:
                continue

            lr_val = np.nanmean(log_ratio[y:y+PATCH_SIZE, x:x+PATCH_SIZE])

            vi_patch = vi[:, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            means_log = np.array([np.log1p(np.nanmean(vi_patch[t])) for t in range(4)])
            if np.any(np.isnan(means_log)):
                slope = 0.0
                intercept = means_log[~np.isnan(means_log)].mean() if np.any(~np.isnan(means_log)) else 0.0
            else:
                slope, intercept = np.polyfit([0., 1., 2., 3.], means_log, 1)

            predicted_log = intercept + slope * 4.0
            actual_log = np.log1p(np.nanmean(v17[0, y:y+PATCH_SIZE, x:x+PATCH_SIZE]))
            dep_val = actual_log - predicted_log

            patch_coords.append((y, x))
            patch_logratio.append(lr_val)
            patch_momentum.append(dep_val)

    patch_logratio = np.array(patch_logratio)
    patch_momentum = np.array(patch_momentum)

    p33, p67 = np.percentile(patch_logratio, [33, 67])
    labels_logratio = np.digitize(patch_logratio, [p33, p67])

    p33, p67 = np.percentile(patch_momentum, [33, 67])
    labels_momentum = np.digitize(patch_momentum, [p33, p67])

    for row, (name, labels) in enumerate([
        ("Log-ratio terciles", labels_logratio),
        ("Momentum terciles", labels_momentum),
    ]):
        ax = axes[row, col]

        label_map = np.full((H, W), np.nan)
        for (y, x), lab in zip(patch_coords, labels):
            label_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = lab

        ax.imshow(bg, cmap="Greys", vmin=bg_vmin, vmax=bg_vmax)
        im = ax.imshow(label_map, cmap=label_cmap, norm=label_norm, alpha=0.45)

        ax.set_title(f"{country.replace('_', ' ')}\n{name}")
        ax.axis("off")

        cbar = plt.colorbar(im, ax=ax, fraction=0.046, ticks=[0, 1, 2])
        cbar.ax.set_yticklabels(["Low", "Mid", "High"])

plt.tight_layout()
plt.show()


## Distribution of change + label thresholds

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle("Nightlight % change distribution (clipped at 99th pct)", fontsize=14, fontweight="bold")

for ax, country in zip(axes.flat, COUNTRIES):
    d          = load_country(country)
    pct_change = (d["viirs_2017"][0] - d["viirs_input"][0]) / (d["viirs_input"][0] + 1e-6) * 100
    flat       = pct_change[np.isfinite(pct_change)]

    p33, p67, p99 = np.percentile(flat, [33, 67, 99])

    ax.hist(flat, bins=80, range=(-200, p99), color="steelblue", edgecolor="none", alpha=0.8)
    ax.axvline(p33, color="orange", linewidth=1.5, label=f"33rd: {p33:.1f}%")
    ax.axvline(p67, color="red",    linewidth=1.5, label=f"67th: {p67:.1f}%")
    ax.set_title(country.replace("_", " "))
    ax.set_xlabel("% change")
    ax.set_ylabel("pixel count")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## Class balance check (patch level)

In [ ]:
def extract_patches(country: str, patch_size: int = None, stride: int = None):
    ps = patch_size if patch_size is not None else PATCH_SIZE
    st = stride     if stride     is not None else STRIDE

    d = load_country(country)

    vi   = d["viirs_input"]                                             # (4, H, W)
    v17  = d["viirs_2017"]                                              # (1, H, W)
    bu   = d["builtup"]                                                 # (1, H, W)
    pop  = np.log1p(np.clip(np.nan_to_num(d["pop"],  nan=0), 0, None)) # (1, H, W)
    elev = np.nan_to_num(d["elev"],  nan=0)                            # (1, H, W)
    slp  = np.nan_to_num(d["slope"], nan=0)                            # (1, H, W)
    prox = np.log1p(np.clip(np.nan_to_num(d["prox"], nan=0), 0, None)) # (1, H, W)
    smod = np.nan_to_num(d["smod"], nan=0)                             # (1, H, W)

    log_ratio = np.log1p(v17[0]) - np.log1p(vi[0])
    H, W = vi.shape[1], vi.shape[2]

    patches, patch_logratio, patch_momentum = [], [], []
    for y in range(0, H - ps, st):
        for x in range(0, W - ps, st):
            v17_patch = v17[0, y:y+ps, x:x+ps]
            if np.isnan(v17_patch).mean() > NAN_THRESH:
                continue
            vi2013_patch = vi[0, y:y+ps, x:x+ps]
            max_patch    = np.maximum(vi2013_patch,
                                      np.nan_to_num(v17_patch, nan=0))
            if np.nanmean(max_patch) < BLANK_THRESH:
                continue

            lr_val   = np.nanmean(log_ratio[y:y+ps, x:x+ps])
            vi_patch = vi[:, y:y+ps, x:x+ps]
            means_log = np.array([np.log1p(np.nanmean(vi_patch[t])) for t in range(4)])
            if np.any(np.isnan(means_log)):
                slope     = 0.0
                intercept = (means_log[~np.isnan(means_log)].mean()
                             if np.any(~np.isnan(means_log)) else 0.0)
            else:
                slope, intercept = np.polyfit([0., 1., 2., 3.], means_log, 1)
            predicted_log = intercept + slope * 4.0
            actual_log    = np.log1p(np.nanmean(v17[0, y:y+ps, x:x+ps]))
            dep_val       = actual_log - predicted_log

            patch_logratio.append(lr_val)
            patch_momentum.append(dep_val)

            p = np.nan_to_num(np.concatenate([
                vi[:,   y:y+ps, x:x+ps],
                bu[:,   y:y+ps, x:x+ps],
                pop[:,  y:y+ps, x:x+ps],
                elev[:, y:y+ps, x:x+ps],
                slp[:,  y:y+ps, x:x+ps],
                prox[:, y:y+ps, x:x+ps],
                smod[:, y:y+ps, x:x+ps],
                v17[:,  y:y+ps, x:x+ps],
            ], axis=0), nan=0.0)
            patches.append(p)

    patches        = np.array(patches)
    patch_logratio = np.array(patch_logratio)
    patch_momentum = np.array(patch_momentum)

    p33, p67 = np.percentile(patch_logratio, [33, 67])
    labels_logratio = np.digitize(patch_logratio, [p33, p67])

    p33, p67 = np.percentile(patch_momentum, [33, 67])
    labels_momentum = np.digitize(patch_momentum, [p33, p67])

    return patches, labels_logratio, labels_momentum


# preview
for country in COUNTRIES:
    patches, labels_lr, labels_mom = extract_patches(country)
    counts = {c: int(np.sum(labels_lr == c)) for c in [0, 1, 2]}
    print(f"{country}: {len(patches)} patches | log-ratio classes {counts}")


## Filter Sensitivity Check

Compare the current endpoint-based blank filter (`max(viirs_2013, viirs_2017)`) against a baseline-only 2013 filter. This is an EDA-only robustness check: if the retained sample changes a lot, we can decide whether the model also needs a sensitivity rerun.

In [ ]:
FILTER_COUNTRIES = COUNTRIES


def patch_filter_stats(country: str, use_baseline_only: bool):
    d = load_country(country)
    vi = d["viirs_input"]
    v17 = d["viirs_2017"]

    H, W = vi.shape[1], vi.shape[2]
    kept = []

    for y in range(0, H - PATCH_SIZE, STRIDE):
        for x in range(0, W - PATCH_SIZE, STRIDE):
            v17_patch = v17[0, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            if np.isnan(v17_patch).mean() > NAN_THRESH:
                continue

            vi2013_patch = vi[0, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            if use_baseline_only:
                score = np.nanmean(vi2013_patch)
            else:
                score = np.nanmean(np.maximum(vi2013_patch, np.nan_to_num(v17_patch, nan=0)))

            if score < BLANK_THRESH:
                continue

            kept.append((y + PATCH_SIZE / 2, x + PATCH_SIZE / 2))

    return np.array(kept) if kept else np.empty((0, 2))


rows = []
for country in FILTER_COUNTRIES:
    keep_endpoint = patch_filter_stats(country, use_baseline_only=False)
    keep_baseline = patch_filter_stats(country, use_baseline_only=True)
    rows.append({
        "country": country,
        "endpoint_filter": len(keep_endpoint),
        "baseline2013_filter": len(keep_baseline),
        "delta": len(keep_baseline) - len(keep_endpoint),
        "pct_change": 100 * (len(keep_baseline) - len(keep_endpoint)) / max(len(keep_endpoint), 1),
    })

filter_df = pd.DataFrame(rows)
print(filter_df.to_string(index=False))


fig, axes = plt.subplots(2, len(FILTER_COUNTRIES), figsize=(3.8 * len(FILTER_COUNTRIES), 8))
fig.suptitle("Patch filter sensitivity: endpoint vs baseline-only", fontsize=14, fontweight="bold")

for col, country in enumerate(FILTER_COUNTRIES):
    d = load_country(country)
    bg = np.log1p(np.clip(d["viirs_input"][0], 0, None))
    valid = bg[np.isfinite(bg)]
    vmin = np.percentile(valid, 1)
    vmax = np.percentile(valid, 99)

    keep_endpoint = patch_filter_stats(country, use_baseline_only=False)
    keep_baseline = patch_filter_stats(country, use_baseline_only=True)

    panels = [
        (axes[0, col], keep_endpoint, "Current endpoint filter"),
        (axes[1, col], keep_baseline, "Baseline-only 2013 filter"),
    ]

    for ax, keep, title in panels:
        ax.imshow(bg, cmap="Greys", vmin=vmin, vmax=vmax)
        if len(keep) > 0:
            ax.scatter(keep[:, 1], keep[:, 0], s=max(8, PATCH_SIZE * 0.9),
                       c="#d62828", alpha=0.65, edgecolors="none")
        n = len(keep)
        ax.set_title(f"{country.replace('_', ' ')}\n{title} ({n} patches)")
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle("Class balance at patch level (32×32, stride 16)", fontsize=14, fontweight="bold")

for col, country in enumerate(COUNTRIES):
    patches, labels_logratio, labels_momentum = extract_patches(country)
    targets = [("Log-ratio", labels_logratio), ("Momentum", labels_momentum)]
    for row, (target_name, labels) in enumerate(targets):
        ax = axes[row, col]
        counts = [int(np.sum(labels == c)) for c in [0, 1, 2]]
        total = sum(counts)
        bars = ax.bar(["Low", "Medium", "High"], counts,
                      color=["steelblue", "orange", "firebrick"])
        for bar, count in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{count/total*100:.1f}%", ha="center", va="bottom", fontsize=9)
        ax.set_title(f"{country.replace('_', ' ')}\n{target_name} ({len(patches)} patches)")
        if col == 0:
            ax.set_ylabel("patch count")

plt.tight_layout()
plt.show()


In [ ]:
def count_patches_for_threshold(country: str, blank_thresh: float) -> int:
    d = load_country(country)
    vi  = d["viirs_input"]
    v17 = d["viirs_2017"]
    H, W = vi.shape[1], vi.shape[2]
    n_patches = 0
    for y in range(0, H - PATCH_SIZE, STRIDE):
        for x in range(0, W - PATCH_SIZE, STRIDE):
            v17_patch = v17[0, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            if np.isnan(v17_patch).mean() > NAN_THRESH:
                continue
            vi2013_patch = vi[0, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            max_patch = np.maximum(vi2013_patch, np.nan_to_num(v17_patch, nan=0))
            if np.nanmean(max_patch) < blank_thresh:
                continue
            n_patches += 1
    return n_patches


rows = []
for country in COUNTRIES:
    n_05 = count_patches_for_threshold(country, 0.5)
    n_03 = count_patches_for_threshold(country, 0.3)
    gain = n_03 - n_05
    pct_gain = 100 * gain / n_05 if n_05 > 0 else np.nan
    rows.append({
        "Country":       country.replace("_", " "),
        "Patches @ 0.5": n_05,
        "Patches @ 0.3": n_03,
        "Gain":          gain,
        "% Gain":        round(pct_gain, 1) if np.isfinite(pct_gain) else np.nan,
    })

patch_compare = pd.DataFrame(rows)
print(patch_compare.to_string(index=False))


## New channel validation

Check that elevation, slope, and infrastructure proximity have sensible relationships with nightlight level.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Elevation vs VIIRS 2017 (sampled pixels, colored by slope)", fontsize=14, fontweight="bold")

for ax, country in zip(axes.flat, COUNTRIES):
    d     = load_country(country)
    ntl   = np.log1p(d["viirs_2017"][0].flatten())
    elev  = d["elev"][0].flatten()
    slope = d["slope"][0].flatten()

    mask = np.isfinite(ntl) & np.isfinite(elev) & np.isfinite(slope)
    ntl, elev, slope = ntl[mask], elev[mask], slope[mask]

    idx = np.random.choice(len(ntl), size=min(5000, len(ntl)), replace=False)
    sc  = ax.scatter(elev[idx], ntl[idx], c=slope[idx], cmap="hot_r",
                     s=2, alpha=0.4, vmin=0, vmax=30)
    plt.colorbar(sc, ax=ax, label="slope (°)")
    ax.set_xlabel("elevation (m)")
    ax.set_ylabel("log1p(VIIRS 2017)")
    ax.set_title(country.replace("_", " "))

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Infrastructure proximity vs VIIRS 2017", fontsize=14, fontweight="bold")

for ax, country in zip(axes.flat, COUNTRIES):
    d    = load_country(country)
    ntl  = np.log1p(d["viirs_2017"][0].flatten())
    prox = np.log1p(np.nan_to_num(d["prox"][0].flatten(), nan=0))

    mask = np.isfinite(ntl) & np.isfinite(prox)
    ntl, prox = ntl[mask], prox[mask]

    idx = np.random.choice(len(ntl), size=min(5000, len(ntl)), replace=False)
    ax.scatter(prox[idx], ntl[idx], s=2, alpha=0.3, color="steelblue")
    ax.set_xlabel("log1p(infra proximity)")
    ax.set_ylabel("log1p(VIIRS 2017)")
    ax.set_title(country.replace("_", " "))

plt.tight_layout()
plt.show()


## Patch extraction

## Patch Size Comparison

Quick sensitivity check for `8x8`, `32x32`, and `64x64` patches, using a stride of half the patch size in each case.

In [ ]:
PATCH_SIZES = [8, 32, 64]


def patch_stats_for_size(country: str, patch_size: int):
    stride = patch_size // 2
    d = load_country(country)
    vi  = d["viirs_input"]
    v17 = d["viirs_2017"]
    H, W = vi.shape[1], vi.shape[2]
    log_ratio = np.log1p(v17[0]) - np.log1p(vi[0])
    patch_logratio = []
    patch_momentum = []
    for y in range(0, H - patch_size, stride):
        for x in range(0, W - patch_size, stride):
            v17_patch = v17[0, y:y+patch_size, x:x+patch_size]
            if np.isnan(v17_patch).mean() > NAN_THRESH:
                continue
            vi2013_patch = vi[0, y:y+patch_size, x:x+patch_size]
            max_patch    = np.maximum(vi2013_patch,
                                      np.nan_to_num(v17[0, y:y+patch_size, x:x+patch_size], nan=0))
            if np.nanmean(max_patch) < BLANK_THRESH:
                continue
            lr_val = np.nanmean(log_ratio[y:y+patch_size, x:x+patch_size])
            vi_patch = vi[:, y:y+patch_size, x:x+patch_size]
            means_log = np.array([np.log1p(np.nanmean(vi_patch[t])) for t in range(4)])
            if np.any(np.isnan(means_log)):
                slope = 0.0
                intercept = means_log[~np.isnan(means_log)].mean() if np.any(~np.isnan(means_log)) else 0.0
            else:
                slope, intercept = np.polyfit([0., 1., 2., 3.], means_log, 1)
            predicted_log = intercept + slope * 4.0
            actual_log = np.log1p(np.nanmean(v17[0, y:y+patch_size, x:x+patch_size]))
            dep_val = actual_log - predicted_log
            patch_logratio.append(lr_val)
            patch_momentum.append(dep_val)
    patch_logratio = np.array(patch_logratio)
    patch_momentum = np.array(patch_momentum)
    p33, p67 = np.percentile(patch_logratio, [33, 67])
    labels_logratio = np.digitize(patch_logratio, [p33, p67])
    p33, p67 = np.percentile(patch_momentum, [33, 67])
    labels_momentum = np.digitize(patch_momentum, [p33, p67])
    return {
        "n_patches": len(patch_logratio),
        "logratio_counts": [int(np.sum(labels_logratio == c)) for c in [0, 1, 2]],
        "momentum_counts": [int(np.sum(labels_momentum == c)) for c in [0, 1, 2]],
    }


rows = []
for country in COUNTRIES:
    for patch_size in PATCH_SIZES:
        stats = patch_stats_for_size(country, patch_size)
        rows.append({
            "Country":      country.replace("_", " "),
            "Patch size":   f"{patch_size}x{patch_size}",
            "Stride":       patch_size // 2,
            "Patches kept": stats["n_patches"],
            "Log Low":      stats["logratio_counts"][0],
            "Log Mid":      stats["logratio_counts"][1],
            "Log High":     stats["logratio_counts"][2],
        })

patch_size_compare = pd.DataFrame(rows)
print(patch_size_compare.to_string(index=False))


## Kosovo And Serbia Patch Maps

Visual comparison of retained patches for `8x8`, `32x32`, and `64x64`, shown separately for Kosovo and Serbia over the 2013 VIIRS background.

In [ ]:
COMPARE_COUNTRIES = ["Kosovo", "Serbia"]
COMPARE_PATCH_SIZES = [8, 32, 64]


def retained_patch_centers(country: str, patch_size: int):
    stride = patch_size // 2
    d = load_country(country)
    vi = d["viirs_input"]
    v17 = d["viirs_2017"]
    H, W = vi.shape[1], vi.shape[2]
    centers_y, centers_x = [], []
    for y in range(0, H - patch_size, stride):
        for x in range(0, W - patch_size, stride):
            v17_patch = v17[0, y:y+patch_size, x:x+patch_size]
            if np.isnan(v17_patch).mean() > NAN_THRESH:
                continue
            vi2013_patch = vi[0, y:y+patch_size, x:x+patch_size]
            max_patch    = np.maximum(vi2013_patch,
                                      np.nan_to_num(v17[0, y:y+patch_size, x:x+patch_size], nan=0))
            if np.nanmean(max_patch) < BLANK_THRESH:
                continue
            centers_y.append(y + patch_size / 2)
            centers_x.append(x + patch_size / 2)
    return d, np.array(centers_y), np.array(centers_x)


fig, axes = plt.subplots(len(COMPARE_COUNTRIES), len(COMPARE_PATCH_SIZES), figsize=(15, 8))
fig.suptitle("Candidate vs retained patch centers", fontsize=14, fontweight="bold")

for row, country in enumerate(COMPARE_COUNTRIES):
    for col, patch_size in enumerate(COMPARE_PATCH_SIZES):
        ax = axes[row, col]
        d, cy, cx = retained_patch_centers(country, patch_size)
        bg = np.log1p(np.clip(d["viirs_input"][0], 0, None))
        valid = bg[np.isfinite(bg)]
        vmin = np.percentile(valid, 1)
        vmax = np.percentile(valid, 99)
        ax.imshow(bg, cmap="Greys", vmin=vmin, vmax=vmax)
        ax.scatter(cx, cy, s=max(8, patch_size * 1.2), c="#d62828", alpha=0.65, edgecolors="none")
        ax.set_title(f"{country} {patch_size}x{patch_size} ({len(cx)} patches)")
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from matplotlib import colors
from matplotlib.patches import Patch
from skimage import measure

COMPARE_COUNTRIES  = ["Kosovo", "Serbia"]
COMPARE_PATCH_SIZES = [8, 32, 64]


def patch_center_sets(country: str, patch_size: int):
    stride = patch_size // 2
    d = load_country(country)
    vi = d["viirs_input"]
    v17 = d["viirs_2017"]
    H, W = vi.shape[1], vi.shape[2]
    all_y, all_x, keep_y, keep_x = [], [], [], []
    for y in range(0, H - patch_size, stride):
        for x in range(0, W - patch_size, stride):
            cy = y + patch_size / 2
            cx = x + patch_size / 2
            all_y.append(cy); all_x.append(cx)
            v17_patch = v17[0, y:y+patch_size, x:x+patch_size]
            if np.isnan(v17_patch).mean() > NAN_THRESH:
                continue
            vi2013_patch = vi[0, y:y+patch_size, x:x+patch_size]
            if np.nanmean(vi2013_patch) < BLANK_THRESH:
                continue
            keep_y.append(cy); keep_x.append(cx)
    return d, np.array(all_y), np.array(all_x), np.array(keep_y), np.array(keep_x)


results_map = {}
max_h, max_w = 0, 0
for country in COMPARE_COUNTRIES:
    for patch_size in COMPARE_PATCH_SIZES:
        d, all_y, all_x, keep_y, keep_x = patch_center_sets(country, patch_size)
        results_map[(country, patch_size)] = (d, all_y, all_x, keep_y, keep_x)
        H, W = d["viirs_input"][0].shape
        max_h = max(max_h, H); max_w = max(max_w, W)

fig, axes = plt.subplots(len(COMPARE_COUNTRIES), len(COMPARE_PATCH_SIZES), figsize=(15, 8))
fig.suptitle("Candidate vs retained patch centers", fontsize=14, fontweight="bold")

for row, country in enumerate(COMPARE_COUNTRIES):
    for col, patch_size in enumerate(COMPARE_PATCH_SIZES):
        ax = axes[row, col]
        d, all_y, all_x, keep_y, keep_x = results_map[(country, patch_size)]
        bg = np.log1p(np.clip(d["viirs_input"][0], 0, None))
        valid = bg[np.isfinite(bg)]
        vmin = np.percentile(valid, 1); vmax = np.percentile(valid, 99)
        H, W = bg.shape
        y0 = (max_h - H) / 2; x0 = (max_w - W) / 2
        extent = [x0, x0 + W, y0 + H, y0]
        ax.imshow(bg, cmap="Greys", vmin=vmin, vmax=vmax, extent=extent)
        footprint = np.isfinite(d["viirs_input"][0])
        for contour in measure.find_contours(footprint.astype(float), 0.5):
            ys, xs = contour[:, 0] + y0, contour[:, 1] + x0
            ax.plot(xs, ys, color="black", linewidth=0.8, alpha=0.35)
        ax.scatter(all_x + x0, all_y + y0, s=max(5, patch_size * 0.8),
                   c="lightgray", alpha=0.35, edgecolors="none")
        ax.scatter(keep_x + x0, keep_y + y0, s=max(6, patch_size * 0.9),
                   c="#d62828", alpha=0.7, edgecolors="none")
        ax.set_xlim(0, max_w); ax.set_ylim(max_h, 0)
        ax.set_aspect("equal")
        ax.set_title(
            f"{country} {patch_size}x{patch_size}\n"
            f"retained {len(keep_x)} / {len(all_x)}"
        )
        ax.axis("off")

legend_handles = [
    Patch(facecolor="lightgray", edgecolor="none", alpha=0.5, label="All candidate centers"),
    Patch(facecolor="#d62828", edgecolor="none", alpha=0.7, label="Retained centers"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


## Summary table

In [ ]:
rows = []
for country in COUNTRIES:
    d               = load_country(country)
    patches, labels_logratio, labels_momentum = extract_patches(country)
    pct_change      = (d["viirs_2017"][0] - d["viirs_input"][0]) / (d["viirs_input"][0] + 1e-6) * 100
    rows.append({
        "Country":          country.replace("_", " "),
        "Raster (H×W)":     f'{d['viirs_input'].shape[1]}×{d['viirs_input'].shape[2]}',
        "Total patches":    len(patches),
        "Class 0 (Low)":    int(np.sum(labels_logratio == 0)),
        "Class 1 (Med)":    int(np.sum(labels_logratio == 1)),
        "Class 2 (High)":   int(np.sum(labels_logratio == 2)),
        "Mean NTL 2013":    round(float(np.nanmean(d["viirs_input"][0])), 3),
        "Mean NTL 2017":    round(float(np.nanmean(d["viirs_2017"][0])), 3),
        "Median % change":  round(float(np.nanmedian(pct_change[np.isfinite(pct_change)])), 1),
        "Mean elevation":   round(float(np.nanmean(d["elev"][0])), 1),
        "Mean slope":       round(float(np.nanmean(d["slope"][0])), 2),
    })

summary = pd.DataFrame(rows)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)
print(summary.to_string(index=False))


## Per-channel normalisation stats

In [ ]:
all_patches_list = []
for country in COUNTRIES:
    patches, _, _ = extract_patches(country)
    all_patches_list.append(patches)
all_patches_arr = np.concatenate(all_patches_list, axis=0)

channel_mean = all_patches_arr.mean(axis=(0, 2, 3))
channel_std  = all_patches_arr.std(axis=(0, 2, 3))

CHANNEL_NAMES_FULL = [
    "VIIRS 2013", "VIIRS 2014", "VIIRS 2015", "VIIRS 2016",
    "Built-up fraction", "Population (log1p)",
    "Elevation", "Slope", "Infra proximity (log1p)",
    "SMOD code", "VIIRS 2017 [target]",
]

print(f"Total patches: {len(all_patches_arr)}")
print(f"\n{'Channel':<28} {'Mean':>8} {'Std':>8}")
print("-" * 48)
for name, m, s in zip(CHANNEL_NAMES_FULL, channel_mean, channel_std):
    print(f"{name:<28} {m:>8.4f} {s:>8.4f}")

print("\nNote: normalisation in model_v2.ipynb is recomputed per fold "
      "using training countries only (no leakage).")


## Save patches to disk

In [ ]:
# Save patches at three scales so model_v2 can compare them.
# Each scale gets its own directory: data/patches_v4_8x8/, _32x32/, _64x64/

SAVE_CONFIGS = [
    (8,  4),    # (patch_size, stride)
    (32, 16),
    (64, 32),
]

for ps, st in SAVE_CONFIGS:
    pdir = Path(f"data/patches_v4_{ps}x{ps}")
    pdir.mkdir(parents=True, exist_ok=True)
    totals = []
    for country in COUNTRIES:
        patches, labels_lr, labels_mom = extract_patches(country, patch_size=ps, stride=st)
        np.save(pdir / f"{country}_patches.npy",          patches)
        np.save(pdir / f"{country}_labels_logratio.npy",  labels_lr)
        np.save(pdir / f"{country}_labels_momentum.npy",  labels_mom)
        totals.append(len(patches))
        print(f"  {ps}x{ps} {country}: {patches.shape}")
    print(f"  → {ps}x{ps} total: {sum(totals)} patches\n")

print("Done.")
